In [ ]:

!pip install mltu==1.2.5


!pip install tqdm==4.65.0

!pip install tf2onnx==1.16.1

In [ ]:
import os
from datetime import datetime

from mltu.configs import BaseModelConfigs

class ModelConfigs(BaseModelConfigs):
    def __init__(self):
        super().__init__()
        self.model_path = os.path.join("/kaggle/working/", datetime.strftime(datetime.now(), "%Y%m%d%H%M"))
        self.vocab = ""
        self.height = 96
        self.width = 1408
        self.max_text_length = 0
        self.batch_size = 32
        self.learning_rate = 0.0005
        self.train_epochs = 5
        self.train_workers = 20

In [ ]:
import keras
print("Keras Version:", keras.__version__)
print(keras.__file__)  # Shows if it's from TensorFlow or standalone


In [ ]:
from keras import layers
from keras.models import Model

from mltu.tensorflow.model_utils import residual_block


def train_model(input_dim, output_dim, activation="leaky_relu", dropout=0.2):
    
    inputs = layers.Input(shape=input_dim, name="input")

    # normalize images here instead in preprocessing step
    input = layers.Lambda(lambda x: x / 255)(inputs)

    x1 = residual_block(input, 32, activation=activation, skip_conv=True, strides=1, dropout=dropout)

    x2 = residual_block(x1, 32, activation=activation, skip_conv=True, strides=2, dropout=dropout)
    x3 = residual_block(x2, 32, activation=activation, skip_conv=False, strides=1, dropout=dropout)

    x4 = residual_block(x3, 64, activation=activation, skip_conv=True, strides=2, dropout=dropout)
    x5 = residual_block(x4, 64, activation=activation, skip_conv=False, strides=1, dropout=dropout)

    x6 = residual_block(x5, 128, activation=activation, skip_conv=True, strides=2, dropout=dropout)
    x7 = residual_block(x6, 128, activation=activation, skip_conv=True, strides=1, dropout=dropout)

    x8 = residual_block(x7, 128, activation=activation, skip_conv=True, strides=2, dropout=dropout)
    x9 = residual_block(x8, 128, activation=activation, skip_conv=False, strides=1, dropout=dropout)

    squeezed = layers.Reshape((x9.shape[-3] * x9.shape[-2], x9.shape[-1]))(x9)

    blstm = layers.Bidirectional(layers.LSTM(256, return_sequences=True))(squeezed)
    blstm = layers.Dropout(dropout)(blstm)

    blstm = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(blstm)
    blstm = layers.Dropout(dropout)(blstm)

    output = layers.Dense(output_dim + 1, activation="softmax", name="output")(blstm)

    model = Model(inputs=inputs, outputs=output)
    return model

In [ ]:
import tensorflow as tf
try: [tf.config.experimental.set_memory_growth(gpu, True) for gpu in tf.config.experimental.list_physical_devices("GPU")]
except: pass

from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, TensorBoard

from mltu.preprocessors import ImageReader
from mltu.transformers import ImageResizer, LabelIndexer, LabelPadding, ImageShowCV2
from mltu.augmentors import RandomBrightness, RandomRotate, RandomErodeDilate, RandomSharpen
from mltu.annotations.images import CVImage

from mltu.tensorflow.dataProvider import DataProvider
from mltu.tensorflow.losses import CTCloss
from mltu.tensorflow.callbacks import Model2onnx, TrainLogger
from mltu.tensorflow.metrics import CERMetric, WERMetric



import os
from tqdm import tqdm

# Must download and extract datasets manually from https://fki.tic.heia-fr.ch/databases/download-the-iam-handwriting-database to Datasets\IAM_Sentences
sentences_txt_path = os.path.join("/kaggle/input/scentence-bbb/sentences.txt")
sentences_folder_path = os.path.join("/kaggle/input/scentence-bbb/mm/mm")

dataset, vocab, max_len = [], set(), 0
words = open(sentences_txt_path, "r").readlines()
for line in tqdm(words):
    if line.startswith("#"):
        continue

    line_split = line.split(" ")
    if line_split[2] == "err":
        continue

    folder1 = line_split[0][:3]
    folder2 = "-".join(line_split[0].split("-")[:2])
    file_name = line_split[0] + ".png"
    label = line_split[-1].rstrip("\n")

    # replace "|" with " " in label
    label = label.replace("|", " ")

    rel_path = os.path.join(sentences_folder_path, folder1, folder2, file_name)
    if not os.path.exists(rel_path):
        print(f"File not found: {rel_path}")
        continue

    dataset.append([rel_path, label])
    vocab.update(list(label))
    max_len = max(max_len, len(label))

# Create a ModelConfigs object to store model configurations
configs = ModelConfigs()

# Save vocab and maximum text length to configs
configs.vocab = "".join(vocab)
configs.max_text_length = max_len
configs.save()

# Create a data provider for the dataset
data_provider = DataProvider(
    dataset=dataset,
    skip_validation=True,
    batch_size=configs.batch_size,
    data_preprocessors=[ImageReader(CVImage)],
    transformers=[
        ImageResizer(configs.width, configs.height, keep_aspect_ratio=True),
        LabelIndexer(configs.vocab),
        LabelPadding(max_word_length=configs.max_text_length, padding_value=len(configs.vocab)),
        ],
)

# Split the dataset into training and validation sets
train_data_provider, val_data_provider = data_provider.split(split = 0.9)

# Augment training data with random brightness, rotation and erode/dilate
train_data_provider.augmentors = [
    RandomBrightness(), 
    RandomErodeDilate(),
    RandomSharpen(),
    ]

# Creating TensorFlow model architecture
model = train_model(
    input_dim = (configs.height, configs.width, 3),
    output_dim = len(configs.vocab),
)




In [ ]:
??leaky_relu.py

In [ ]:
# Compile the model and print summary
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=configs.learning_rate), 
    loss=CTCloss(), 
    metrics=[
        CERMetric(vocabulary=configs.vocab),
        WERMetric(vocabulary=configs.vocab)
        ],
    run_eagerly=False
)


In [ ]:


# Define callbacks
earlystopper = EarlyStopping(monitor="val_CER", patience=20, verbose=1, mode="min")
checkpoint = ModelCheckpoint(f"{configs.model_path}/model.keras", monitor="val_CER", verbose=1, save_best_only=True, mode="min")
trainLogger = TrainLogger(configs.model_path)
tb_callback = TensorBoard(f"{configs.model_path}/logs", update_freq=1)
reduceLROnPlat = ReduceLROnPlateau(monitor="val_CER", factor=0.9, min_delta=1e-10, patience=5, verbose=1, mode="auto")
model2onnx = Model2onnx(f"{configs.model_path}/model.keras")

# Train the model
model.fit(
    train_data_provider,
    validation_data=val_data_provider,
    epochs=1,
    callbacks=[earlystopper, checkpoint, trainLogger, reduceLROnPlat, tb_callback, model2onnx],
    
)

# Save training and validation datasets as csv files
train_data_provider.to_csv(os.path.join(configs.model_path, "train.csv"))
val_data_provider.to_csv(os.path.join(configs.model_path, "val.csv"))

In [ ]:
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from keras.models import load_model
from mltu.utils.text_utils import ctc_decoder, get_cer, get_wer

# Load your trained model (.keras)


# Define your character set (should match what you trained with)
char_list = list("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789")

# Define model input size from model shape
IMG_HEIGHT = model.input_shape[1]
IMG_WIDTH = model.input_shape[2]

# Resize function to keep aspect ratio and pad to fit input size
def resize_with_padding(image, target_width, target_height):
    h, w = image.shape[:2]
    scale = min(target_width / w, target_height / h)
    resized_w = int(w * scale)
    resized_h = int(h * scale)
    resized = cv2.resize(image, (resized_w, resized_h))

    pad_width = target_width - resized_w
    pad_height = target_height - resized_h

    top = pad_height // 2
    bottom = pad_height - top
    left = pad_width // 2
    right = pad_width - left

    padded = cv2.copyMakeBorder(resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(255, 255, 255))
    return padded

# Preprocess image to match model input
def preprocess_image(image):
    image = resize_with_padding(image, IMG_WIDTH, IMG_HEIGHT)
    if len(image.shape) == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    image = image / 255.0
    return np.expand_dims(image, axis=0).astype(np.float32)

# Prediction function
def predict(image):
    preds = model.predict(preprocess_image(image))
    return ctc_decoder(preds, char_list)[0]

# Load validation data
df = pd.read_csv("/kaggle/working/202504211111/configs.yaml").values.tolist()

# Run inference and evaluate
accum_cer, accum_wer = [], []
for image_path, label in tqdm(df):
    image = cv2.imread(image_path.replace("\\", "/"), cv2.IMREAD_GRAYSCALE)
    if image is None:
        print(f"Error loading image: {image_path}")
        continue

    prediction_text = predict(image)
    cer = get_cer(prediction_text, label)
    wer = get_wer(prediction_text, label)

    print(f"Image: {image_path}\nLabel: {label}\nPrediction: {prediction_text}\nCER: {cer:.4f}; WER: {wer:.4f}\n")
    accum_cer.append(cer)
    accum_wer.append(wer)

print(f"Average CER: {np.mean(accum_cer):.4f}, Average WER: {np.mean(accum_wer):.4f}")


In [ ]:
import cv2
import numpy as np
from keras.models import load_model
from mltu.utils.text_utils import ctc_decoder, get_cer, get_wer
from mltu.transformers import ImageResizer

class ImageToWordModel:
    def __init__(self, model_path, char_list):
        self.model = load_model(model_path)
        self.char_list = char_list

    def predict(self, image: np.ndarray):
        # Preprocess image
        image = ImageResizer.resize_maintaining_aspect_ratio(image, *self.model.input_shape[1:3][::-1])
        image = image / 255.0  # Normalize image
        image = np.expand_dims(image, axis=0).astype(np.float32)

        # Make prediction
        preds = self.model.predict(image)
        text = ctc_decoder(preds, self.char_list)[0]
        return text


if __name__ == "__main__":
    import pandas as pd
    from tqdm import tqdm

    # Load model configuration (for vocab)
    configs = BaseModelConfigs.load("/kaggle/working/202504211111/configs.yaml")

    # Load the Keras model
   
    df = pd.read_csv("/kaggle/working/202504211111/val.csv").values.tolist()

    accum_cer, accum_wer = [], []
    for image_path, label in tqdm(df):
        image = cv2.imread(image_path.replace("\\", "/"))

        # Predict text from the image
        prediction_text = model.predict(image)

        # Calculate CER and WER
        cer = get_cer(prediction_text, label)
        wer = get_wer(prediction_text, label)

        # Output results
        print("Image: ", image_path)
        print("Label:", label)
        print("Prediction: ", prediction_text)
        print(f"CER: {cer}; WER: {wer}")

        accum_cer.append(cer)
        accum_wer.append(wer)

        cv2.imshow(prediction_text, image)
        cv2.waitKey(0)
        cv2.destroyAllWindows()

    print(f"Average CER: {np.average(accum_cer)}, Average WER: {np.average(accum_wer)}")


In [ ]:
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from keras.models import load_model
from mltu.utils.text_utils import ctc_decoder, get_cer, get_wer
from mltu.transformers import ImageResizer

# Define character set
char_list = list("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789")

# Load model


# Preprocess image
def preprocess_image(image):
    image = ImageResizer.resize_maintaining_aspect_ratio(image, *model.input_shape[1:3][::-1])
    if len(image.shape) == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    image = image / 255.0
    return np.expand_dims(image, axis=0).astype(np.float32)

# Prediction function
def predict(image):
    preds = model.predict(preprocess_image(image))
    return ctc_decoder(preds, char_list)[0]

# Load validation data
df = pd.read_csv("/kaggle/working/202503311314/val.csv").values.tolist()

# Run inference and calculate metrics
accum_cer, accum_wer = [], []
for image_path, label in tqdm(df):
    image = cv2.imread(image_path.replace("\\", "/"), cv2.IMREAD_GRAYSCALE)
    if image is None:
        print(f"Error loading image: {image_path}")
        continue

    prediction_text = predict(image)
    cer = get_cer(prediction_text, label)
    wer = get_wer(prediction_text, label)

    print(f"Image: {image_path}\nLabel: {label}\nPrediction: {prediction_text}\nCER: {cer}; WER: {wer}\n")
    accum_cer.append(cer)
    accum_wer.append(wer)

print(f"Average CER: {np.mean(accum_cer):.4f}, Average WER: {np.mean(accum_wer):.4f}")


In [ ]:
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from mltu.utils.text_utils import ctc_decoder, get_cer, get_wer
from mltu.transformers import ImageResizer
from keras.models import load_model


char_list = list("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789")

# Preprocess image
def preprocess_image(image):
    image = ImageResizer.resize_maintaining_aspect_ratio(image, *model.input_shape[1:3][::-1])

    # Convert grayscale to RGB if needed
    if len(image.shape) == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)

    return np.expand_dims(image, axis=0).astype(np.float32)

# Prediction function
def predict(image):
    preds = model.predict(preprocess_image(image))
    return ctc_decoder(preds, char_list)[0]

# Load dataset
df = pd.read_csv("/kaggle/working/202503311314/val.csv").values.tolist()

# Run inference and compute metrics
accum_cer, accum_wer = [], []
for image_path, label in tqdm(df):
    image = cv2.imread(image_path.replace("\\", "/"), cv2.IMREAD_GRAYSCALE)

    if image is None:
        print(f"Error loading image: {image_path}")
        continue

    prediction_text = predict(image)

    cer, wer = get_cer(prediction_text, label), get_wer(prediction_text, label)
    print(f"Image: {image_path}\nLabel: {label}\nPrediction: {prediction_text}\nCER: {cer}; WER: {wer}\n")

    accum_cer.append(cer)
    accum_wer.append(wer)

print(f"Average CER: {np.mean(accum_cer):.4f}, Average WER: {np.mean(accum_wer):.4f}")


In [ ]:
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from mltu.utils.text_utils import ctc_decoder, get_cer, get_wer
from mltu.transformers import ImageResizer
from keras.models import load_model

# Load model

char_list = list("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789")

# Preprocess image
def preprocess_image(image):
    image = ImageResizer.resize_maintaining_aspect_ratio(image, *model.input_shape[1:3][::-1])
    
    # Convert grayscale to RGB if needed
    if len(image.shape) == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    
    return np.expand_dims(image, axis=0).astype(np.float32)

# Prediction function
def predict(image):
    preds = model.predict(preprocess_image(image))
    return ctc_decoder(preds, char_list)[0]

# Load dataset
df = pd.read_csv("/kaggle/working/202503311314/val.csv").values.tolist()

# Run inference and compute metrics
accum_cer, accum_wer = [], []
for image_path, label in tqdm(df):
    image = cv2.imread(image_path.replace("\\", "/"), cv2.IMREAD_GRAYSCALE)
    prediction_text = predict(image)

    cer, wer = get_cer(prediction_text, label), get_wer(prediction_text, label)
    print(f"Image: {image_path}\nLabel: {label}\nPrediction: {prediction_text}\nCER: {cer}; WER: {wer}\n")

    accum_cer.append(cer)
    accum_wer.append(wer)

print(f"Average CER: {np.average(accum_cer)}, Average WER: {np.average(accum_wer)}")


In [ ]:
from tensorflow.keras.utils import Sequence

class MyDataGenerator(Sequence):
    def __init__(self, data, batch_size):
        self.data = data
        self.batch_size = batch_size

    def __len__(self):
        return int(len(self.data) / self.batch_size)

    def __getitem__(self, idx):
        batch_x = self.data[idx * self.batch_size:(idx + 1) * self.batch_size]
        return batch_x  # Ensure this returns correct (X, Y) format

train_data_provider = MyDataGenerator(train_data_provider, batch_size=32)
val_data_provider = MyDataGenerator(val_data_provider, batch_size=32)


In [ ]:
model.fit(
    train_data_provider,
    validation_data=val_data_provider,
    epochs=1,
    callbacks=[earlystopper, checkpoint, trainLogger, reduceLROnPlat, tb_callback, model2onnx],
    workers=configs.train_workers,  # Only valid if using Sequence
    use_multiprocessing=True
)
# Save training and validation datasets as csv files
train_data_provider.to_csv(os.path.join(configs.model_path, "train.csv"))
val_data_provider.to_csv(os.path.join(configs.model_path, "val.csv"))


In [ ]:
methods = [func for func in dir(Model) if callable(getattr(Model, func)) and not func.startswith("__")]

print(methods)

In [ ]:
model.load_weights(f"{configs.model_path}/model.weights.h5")


In [ ]:
model.save(f"{configs.model_path}/model.h5", save_format="h5")


In [ ]:
model2onnx = Model2onnx(f"{configs.model_path}/model.h5")

In [ ]:
Model2onnx.model2onnx(model,"/kaggle/working/202411262354/model.onnx")

In [ ]:
import cv2
import typing
import numpy as np

from mltu.inferenceModel import OnnxInferenceModel
from mltu.utils.text_utils import ctc_decoder, get_cer, get_wer
from mltu.transformers import ImageResizer

class ImageToWordModel(OnnxInferenceModel):
    def __init__(self, char_list: typing.Union[str, list], *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.char_list = char_list

    def predict(self, image: np.ndarray):
        image = ImageResizer.resize_maintaining_aspect_ratio(image, *self.input_shapes[0][1:3][::-1])

        image_pred = np.expand_dims(image, axis=0).astype(np.float32)

        preds = self.model.run(self.output_names, {self.input_names[0]: image_pred})[0]

        text = ctc_decoder(preds, self.char_list)[0]

        return text

if __name__ == "__main__":
    import pandas as pd
    from tqdm import tqdm
    from mltu.configs import BaseModelConfigs
    




    configs = BaseModelConfigs.load("/kaggle/working/202503161147/configs.yaml")

    model = ImageToWordModel(model_path="/kaggle/working/202503161147/model.keras", char_list=configs.vocab)

    df = pd.read_csv("/kaggle/working/202503161147/val.csv").values.tolist()

    accum_cer, accum_wer = [], []
    for image_path, label in tqdm(df):
        image = cv2.imread(image_path.replace("\\", "/"))

        prediction_text = model.predict(image)

        cer = get_cer(prediction_text, label)
        wer = get_wer(prediction_text, label)
        print("Image: ", image_path)
        print("Label:", label)
        print("Prediction: ", prediction_text)
        print(f"CER: {cer}; WER: {wer}")

        accum_cer.append(cer)
        accum_wer.append(wer)

        cv2.imshow(prediction_text, image)
        cv2.waitKey(0)
        cv2.destroyAllWindows()

    print(f"Average CER: {np.average(accum_cer)}, Average WER: {np.average(accum_wer)}")

In [ ]:
print(configs.model_path)

In [ ]:
import tf2onnx
h5_model_path = f"{configs.model_path}/model.h5"
model = tf.keras.models.load_model(h5_model_path, compile=False)

# Convert the model to ONNX format
onnx_model_path = f"{configs.model_path}/model_1.onnx"
spec = (tf.TensorSpec((None, *model.input.shape[1:]), tf.float32, name="input"),)
onnx_model, _ = tf2onnx.convert.from_keras(model, input_signature=spec, opset=13)

# Save the ONNX model
with open(onnx_model_path, "wb") as f:
    f.write(onnx_model.SerializeToString())

print(f"Model successfully converted and saved as {onnx_model_path}")